# Chapitre 12 - Exercices 12.1 à 12.5

* 12.1 : équivalence entre la forme « perte charnière + pénalité » (12.25) et le problème
  quadratique (12.8) ;
* 12.2 : la forme à noyau (12.29) est (12.25) pour un noyau particulier ;
* 12.3 : régression à noyau quand on ne pénalise pas la constante ;
* 12.4 : distances dans l'espace des variables canoniques de la LDA ;
* 12.5 : classification de phonèmes par prototypes lisses (k-moyennes sur une base de
  splines).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch12_svm import (SVMClassifier, hinge_loss, rbf_kernel, polynomial_kernel,
                          poly2_features, LDA, kmeans, natural_spline_basis)
set_style()

## Exercice 12.1 : Équivalence de (12.25) et (12.8)

**Énoncé.** Montrer que les critères (12.25) et (12.8) sont équivalents, c'est-à-dire que

$$\text{(12.8)}\quad \min_{\beta,\beta_0,\xi}\ \tfrac12\|\beta\|^2 + C\sum_i\xi_i\ \ \text{s.c.}\ \ \xi_i\ge0,\ y_i(x_i^T\beta+\beta_0)\ge1-\xi_i$$

et

$$\text{(12.25)}\quad \min_{\beta,\beta_0}\ \sum_i\left[1 - y_i(x_i^T\beta + \beta_0)\right]_+ + \frac\lambda2\|\beta\|^2$$

ont les mêmes solutions $(\hat\beta,\hat\beta_0)$ lorsque $\lambda = 1/C$.

**Solution.** *Étape 1 : élimination des variables d'écart.* Fixons $(\beta,\beta_0)$ et
notons $m_i = y_i(x_i^T\beta + \beta_0)$. Les contraintes sur $\xi_i$ sont $\xi_i\ge0$ et
$\xi_i \ge 1 - m_i$, soit $\xi_i \ge \max(0, 1-m_i) = [1-m_i]_+$. Comme l'objectif est
strictement croissant en chaque $\xi_i$ (coefficient $C > 0$), le meilleur choix est la plus
petite valeur admissible : $\xi_i = [1 - m_i]_+$.

*Étape 2 : problème réduit.* Minimiser (12.8) revient donc à minimiser, en $(\beta,\beta_0)$
seulement,

$$\Phi(\beta,\beta_0) = \tfrac12\|\beta\|^2 + C\sum_i[1-m_i]_+ .$$

En effet, si $(\beta^\star,\beta_0^\star,\xi^\star)$ minimise (12.8), alors pour tout
$(\beta,\beta_0)$ : $\Phi(\beta^\star,\beta_0^\star)\le$ valeur de (12.8) en
$(\beta^\star,\beta_0^\star,\xi^\star)\le$ valeur de (12.8) en $(\beta,\beta_0,[1-m]_+) = \Phi(\beta,\beta_0)$.
Réciproquement, un minimiseur de $\Phi$ complété par $\xi_i = [1-m_i]_+$ minimise (12.8).

*Étape 3 : changement d'échelle.* Multiplier une fonction par la constante $\lambda = 1/C > 0$
ne change pas ses minimiseurs :

$$\lambda\,\Phi(\beta,\beta_0) = \sum_i[1 - y_i(x_i^T\beta+\beta_0)]_+ + \frac\lambda2\|\beta\|^2,$$

qui est exactement (12.25). Les deux problèmes ont donc les mêmes solutions, et la valeur
optimale de (12.25) vaut $\lambda$ fois celle de (12.8). $\square$

**Vérification numérique.** On résout (12.25) directement, avec un optimiseur générique
(`scipy.optimize.minimize`, méthode SLSQP) appliqué à la forme « épigraphe » sans le terme
quadratique en $\xi$, et on compare au SVM obtenu par le dual.

In [2]:
rng = np.random.RandomState(0)
X = np.vstack([rng.randn(25, 2) + [1, 1], rng.randn(25, 2) - [1, 1]])
y = np.r_[np.ones(25), -np.ones(25)]
C = 0.5
lam = 1 / C

def crit_1225(theta):
    beta, beta0 = theta[:2], theta[2]
    return hinge_loss(y * (X @ beta + beta0)).sum() + lam / 2 * beta @ beta

# (12.25) n'est pas dérivable : on passe par les écarts xi pour SLSQP
N = len(y)
obj = lambda v: v[3:].sum() + lam / 2 * v[:2] @ v[:2]
cons = [{"type": "ineq", "fun": lambda v: v[3:]},
        {"type": "ineq", "fun": lambda v: y * (X @ v[:2] + v[2]) - 1 + v[3:]}]
sol = minimize(obj, np.r_[0, 0, 0, np.ones(N)], constraints=cons, method="SLSQP",
               options={"maxiter": 500, "ftol": 1e-12})
svm = SVMClassifier(C=C, kernel="linear").fit(X, y)
print("solution directe (12.25)  : beta =", sol.x[:2].round(5), " beta0 = %.5f" % sol.x[2])
print("SVM via le dual (12.8)    : beta =", svm.coef_.round(5), " beta0 = %.5f" % svm.b_)
print("critère (12.25) : %.6f  ;  lambda x critère (12.8) : %.6f"
      % (crit_1225(np.r_[svm.coef_, svm.b_]), lam * svm.primal_objective()))

solution directe (12.25)  : beta = [0.78548 0.87657]  beta0 = -0.09282
SVM via le dual (12.8)    : beta = [0.78548 0.87657]  beta0 = -0.09282
critère (12.25) : 10.029449  ;  lambda x critère (12.8) : 10.029449
